### Creat agent with Agent state


#### Agent State is the data an agent maintains while it executes.


In [ ]:
from langchain.agents import AgentState

messages: list[BaseMessage]

In [ ]:
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import after_model

class MyState(AgentState):
    user_id: str
    call_count: int

# after = create  agent (done) -> call
# before = call ->  create agent
@after_model
def increment_call_count(state: MyState, runtime): 
    print("run the middleware")
    current_count = state.get("call_count", 0)
    return {
        "call_count" : current_count + 1,
        "user_id" : "pratik-02"
    }

agent = create_agent(
    model="openai:gpt-5.5",
    state_schema=MyState,
    middleware=[increment_call_count]
)


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain caching in one sentence.",
            }
        ],
        "user_id" : "pratik-01",
        "call_count": 0,

    }
)
print("User ID:", result["user_id"])
print("Call count:", result["call_count"])
print("AI response:", result["messages"][-1].content)

run the middleware
User ID: pratik-02
Call count: 1
AI response: Caching is storing frequently used data in a faster-to-access place so future requests can be served more quickly.


#### messages field grows as a LangChain agent processes a conversation

In [ ]:
# 1 
{
  "messages": [
    {
      "type": "human",
      "content": "What is caching?",
      "id": "msg-human-1"
    },
    {
      "type": "ai",
      "content": "Caching is storing frequently used data in a faster-to-access place so future requests can be served more quickly.",
      "id": "msg-ai-1"
    },
  ]
}
#  2
{
  "messages": [
    {
      "type": "human",
      "content": "What is caching?",
      "id": "msg-human-1"
    },
    {
      "type": "ai",
      "content": "Caching stores frequently accessed data for faster retrieval.",
      "id": "msg-ai-1"
    },
    {
      "type": "human",
      "content": "Give me a real-world example.",
      "id": "msg-human-2"
    },
    {
      "type": "ai",
      "content": "A browser caches images so they load faster when you revisit a website.",
      "id": "msg-ai-2"
    }
  ]
}



### Have not declared user_id in MyState

In [ ]:
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import after_model

class MyState(AgentState):
    user_id: str
    call_count: int

# after = create  agent (done) -> call
# before = call ->  create agent
@after_model
def increment_call_count(state: MyState, runtime): 
    current_count = state.get("call_count", 0)
    return {
        "call_count" : current_count + 1,
        "user_id" : "pratik-02"
    }
# increment_call_count(state, b)
 
agent = create_agent(
    model="openai:gpt-5.5",
    state_schema=MyState,
    middleware=[increment_call_count]
)


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain caching in one sentence.",
            }
        ],
        "user_id" : "pratik-01",
        "call_count": 0,

    }
)
print("User ID:", result["user_id"])
print("Call count:", result["call_count"])
print("AI response:", result["messages"][-1].content)

User ID: pratik-02
Call count: 101
AI response: Caching is storing frequently used data in a faster temporary location so future requests can be served more quickly.


### Return a key that isn't in MyState